In [81]:
import os
import pygmt
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
import numpy as np
import pandas as pd 
import glob 

%load_ext autoreload 
%autoreload 2
%matplotlib inline
import utils
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [82]:
#### this is the only cell that needs changing before running 
main_dir = "model_outputs/two_faults/inplane/"
files_interest = ["Dc_002/output_Dc002_5km/"] 
loading_type = "In plane"
dc = 0.002 # Dc in RSF 
D = 5 # distance between the two parallel faults in kilometers
spin_up = 200  # n of years to omit in analysis - approx time it takes for steady state to be reached

In [ ]:
cell_dict = {
    0.01: (20, 4096),
    0.008: (20, 4096),
    0.0063: (10, 8192),
    0.005: (10, 8192),
    0.0039: (10, 8192),
    0.003: (5, 16384),
    0.002: (5, 16384),
    0.0014: (2.5, 32768),
    0.001: (2.5, 32768) 
} # DX and N2 info for each model from Motorcycle shell scripts 

DX, N2 = cell_dict[dc]
DX = [DX]
N2 = [N2]
cols = ['Index', 'Displacement', 'Col3', 'Col4', 'Col5', 'col', 'cplb','Slip Rate', 'Col7', 'Col8', 'Col9', 'Col10','Col11'] 
# time step conversion
time_step_rate = 20 # --export-netcdf-rate 20 \ ----- I am using the same in all models so this one shouldn't change..
grid_step_rate_horizontal = 4

feature = 'log10v' # tau, slip, log10v
fault1_path = main_dir + "fault-01-" + feature + ".grd"

In [84]:
time_step_rate = 20
grid_step_rate_horizontal = 4 

for i, (file, DXi, N2i) in enumerate(zip(files_interest, DX, N2)):
    file = file
    
    # load time file for times
    file_pattern = main_dir + file + 'patch-01-*.dat'
    matching_files = glob.glob(file_pattern)
    fault_data = pd.read_csv(matching_files[0], sep='\s+', header=None, names=cols)

    # load grid for moment
    feature = 'log10v' # tau, slip, log10v
    fault1_path = main_dir + file + "fault-01-" + feature + ".grd"
    fault2_path = main_dir + file + "fault-02-" + feature + ".grd"
    grid_fault1 = pygmt.load_dataarray(fault1_path, engine='netcdf4')
    grid_fault2 = pygmt.load_dataarray(fault2_path, engine='netcdf4')
    delta_grid = time_step_rate   
    time_grid = fault_data['Index'].iloc[::int(delta_grid)].values
    
    # downsample, otherwise gets massive for the big simulations
    if dc>0.005:
        grid_fault1 = grid_fault1.coarsen(x=2, boundary='trim').mean()
        grid_fault2 = grid_fault2.coarsen(x=2, boundary='trim').mean()
    elif 0.0014 <= dc <= 0.005:
        grid_fault1 = grid_fault1.coarsen(x=3, boundary='trim').mean()
        grid_fault2 = grid_fault2.coarsen(x=3, boundary='trim').mean()
    else:
        grid_fault1 = grid_fault1.coarsen(x=4, boundary='trim').mean()
        grid_fault2 = grid_fault2.coarsen(x=4, boundary='trim').mean()

    # make catalog
    grid1_masked = np.ma.masked_where(grid_fault1 < -3.5, grid_fault1) # subset areas where velocity>seismic slip to isolate events
    grid2_masked = np.ma.masked_where(grid_fault2 < -3.5, grid_fault2) # subset areas where velocity>seismic slip to isolate events
    grid1_masked = grid1_masked.filled(np.nan) # for viz in seaborn fill unmasked areas with nan
    grid2_masked = grid2_masked.filled(np.nan) # for viz in seaborn fill unmasked areas with nan
    timestep_event_fault1, rupture_length_pixels_fault1, group_id_fault1, total_pixels_fault1_xdir,pixel_size = utils.find_events(grid1_masked,DXi,N2i,"No")
    timestep_event_fault2, rupture_length_pixels_fault2, group_id_fault2, total_pixels_fault2_xdir,pixel_size = utils.find_events(grid2_masked,DXi,N2i,"No")
    time_f1 = time_grid[timestep_event_fault1]
    time_f2 = time_grid[timestep_event_fault2]
    rupture_length_fault1 = np.array(rupture_length_pixels_fault1) 
    rupture_length_fault2 = np.array(rupture_length_pixels_fault2) 

    # remove spin-up period 
    time_cutoff = spin_up * 365 * 24 * 60 * 60
    cutoff_index = np.where(time_grid >= time_cutoff)[0][0]
    time_grid = time_grid[cutoff_index:]
    time_f1 = time_f1[time_f1 >= time_cutoff]
    time_f2 = time_f2[time_f2 >= time_cutoff]
    rupture_length_fault1 = rupture_length_fault1[len(rupture_length_fault1) - len(time_f1):]
    rupture_length_fault2 = rupture_length_fault2[len(rupture_length_fault2) - len(time_f2):]

    # remove ruptures that are 4 cells only (1 cell in here because of MTC downsampling) -- artifacts from masking
    ok_size_rupture_idx1 = np.where(rupture_length_fault1 > 3)[0]
    ok_size_rupture_idx2 = np.where(rupture_length_fault2 > 3)[0]
    time_f1 = time_f1[ok_size_rupture_idx1]
    rupture_length_fault1 = rupture_length_fault1[ok_size_rupture_idx1]
    time_f2 = time_f2[ok_size_rupture_idx2]
    rupture_length_fault2 = rupture_length_fault2[ok_size_rupture_idx2]
    
    # remove partial ruptures (ruptures with length < 4.5 km)
    time_f1 = time_f1[rupture_length_fault1 * pixel_size > 4500]
    time_f2 = time_f2[rupture_length_fault2 * pixel_size > 4500]
    print(time_f1/(365 * 24 * 60 * 60) ,time_f2/(365 * 24 * 60 * 60))

    # measure interevent times
    inter_event_times_f1 = utils.measure_interevent_time_f(time_f1) # in seconds
    inter_event_times_years_f1 = inter_event_times_f1 / (365 * 24 * 60 * 60) # in years
    inter_event_times_f2 = utils.measure_interevent_time_f(time_f2) # in seconds
    inter_event_times_years_f2 = inter_event_times_f2 / (365 * 24 * 60 * 60) # in years

    print('Inter-event times fault 1 (years)', inter_event_times_years_f1, 'Inter-event times fault 2 (years)', inter_event_times_years_f2)
    
    times_file =  "code_output_data/interevent_times_two_faults_full_ruptures.csv"
    if not os.path.isfile(times_file):
        cols = ["Loading", "Dc", "D", "Inter-event times fault 1 (seconds)", "Inter-event times fault 1 (years)", "Inter-event times fault 2 (seconds)", "Inter-event times fault 2 (years)"]
        times_database = pd.DataFrame(columns=cols)
        times_database.to_csv(times_file, index=False)
    else:
        times_database = pd.read_csv(times_file)
    existing_combination = times_database[(times_database['Dc'] == dc) & (times_database['D'] == D) & (times_database['Loading'] == loading_type)]
    if not existing_combination.empty:
        times_database = times_database[~((times_database['Dc'] == dc) & (times_database['D'] == D) & (times_database['Loading'] == loading_type))]
        print(f"Removed existing rows with Dc={dc} and D={D}.")

    times_sim_i = pd.DataFrame({
        'Loading': [loading_type],
        'Dc': [dc],
        'D':D,
        'Inter-event times fault 1 (seconds)': [", ".join(map(str, inter_event_times_f1))],
        'Inter-event times fault 1 (years)': [", ".join(map(str, inter_event_times_years_f1))],
        'Inter-event times fault 2 (seconds)': [", ".join(map(str, inter_event_times_f2))],
        'Inter-event times fault 2 (years)': [", ".join(map(str, inter_event_times_years_f2))]
    })
    
    new_rows = pd.concat([times_sim_i], ignore_index=True)
    catalog_df = pd.concat([times_database, new_rows], ignore_index=True)
    catalog_df.to_csv(times_file, index=False)
    print("New row added to interevent time file.csv")

[ 202.65796597  217.83711819  236.53832245  266.04607896  279.95398642
  327.22018872  366.72870391  378.33123459  402.30977302  427.83409416
  462.78812726  507.45661452  539.28996611  556.09727883  598.19044716
  638.05760505  650.53507518  697.93429746  731.34875207  762.77858422
  779.8042963   829.07162123  846.00986291  885.50560565  902.38021769
  922.3322085   945.07376921  990.78185021 1020.35839425 1031.97895419
 1052.39550276 1099.67660674] [ 211.07300307  245.1231027   286.85618623  323.81349281  353.83603886
  365.24585939  402.65964446  419.85124027  463.30778181  479.45951753
  498.49082479  539.01484534  580.77232944  592.13705332  616.42936274
  643.4623976   684.61543086  697.98394874  731.25152646  769.62894107
  812.94234204  825.05468027  846.40724257  885.1889404   923.46138128
  936.40147229  981.43221149  991.41002441 1031.79226372 1046.05214867
 1081.94419211 1094.93728439]
Inter-event times fault 1 (years) [15.17915221 18.70120426 29.50775652 13.90790746 47.26